In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
%pip install catboost

In [ ]:
import pandas as pd
import numpy as np # for random data generation
import matplotlib.pyplot as plt
import os
import torch
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, LabelEncoder #import OneHotEncoder
from sklearn.model_selection import KFold ,StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from tqdm import tqdm
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from lightgbm import LGBMRegressor
from catboost import CatBoostClassifier


In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')

In [ ]:
# Task 2: Write your code here Inspect the first few rows using head():
df.head()

In [ ]:
# Task 3: Write your code here Display dataset information using info():
df.info()

In [ ]:
# Task 4: Write your code here Show statistical description using describe():
df.describe()

In [ ]:
# Task 1: Write your code here:
the_null = df.isnull().sum()
to_drop=the_null[the_null>10000].index.to_list()
df_clean = df.copy()
df_clean =df_clean.drop(columns=to_drop)

In [ ]:
missing_values= df_clean.isnull().sum()[df_clean.isnull().sum()>0]

for col in missing_values.index.tolist():

    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())
df_clean.isnull().sum()[df_clean.isnull().sum()>0]

In [ ]:
# Task 2 Check and remove duplicates if any exist:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df_clean)

In [ ]:
# Task 3 Check and Encode categorical variables if needed:
df_clean.select_dtypes(include=["object"]).columns

In [ ]:
# Task 4 Apply feature scaling to numerical features (Use StandardScaler): Write your code here :
features = df_clean.columns.drop("Target")  # DON'T SCALE THE TARGET

scale = StandardScaler()
df_clean[features] = scale.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 5: Write your code here Check for target imbalance and state if it is imbalanced or not:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Target")

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Target",axis=1)
y = df_clean['Target']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Task 2,3,4,5: Write your code here:
lr_accuracy = []
lr_f1 = []
model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models


  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred, zero_division=0)
  recall = recall_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  lr_accuracy.append(accuracy)
  lr_f1.append(f1)

print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Task 1: Write your code here:

coeffs = {}

coeffs["tree"] = model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(40,24))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here:
X = df_clean["P_2"]
y = df_clean['Target']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Task 2,3,4,5: Write your code here:
lr_accuracy_2 = []
lr_f1_2 = []
model = CatBoostClassifier(verbose=0,n_estimators=320,max_depth=4)

n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models


  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred, zero_division=0)
  recall = recall_score(y_test, y_pred, zero_division=0)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  lr_accuracy_2.append(accuracy)
  lr_f1_2.append(f1)

print(f"  Accuracy:  {np.mean(lr_accuracy_2):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1_2):.4f}")

In [ ]:
np.mean(lr_accuracy_2)==np.mean(lr_accuracy)


In [ ]:
np.mean(lr_f1_2)==np.mean(lr_f1)